# Training scRepresenter

This step involves training both scNET and scGPT, and combining the resulting output. GPU usage is highly recommended.

In [1]:
#required imports and misc.

import os
import scanpy as sc
if os.path.basename(os.getcwd()) == 'scripts':
    os.chdir('..')
from src.main import run_scRepresenter

RUN_NAME = "pbmc3k"

SAVE_DIR = "output/" + RUN_NAME
os.makedirs(SAVE_DIR, exist_ok=True)

Global seed set to 0


## Prepare input data

There are two required inputs to run the scRepresenter pipeline, a scRNA-seq dataset and a gene similarity network. We already processed a scRNA-seq dataset (see the Preprocessing notebook), so we just need to read the file into memory.

For the gene network, we will use the Protein-Protein Interactions graph that we provide. For a guide on how to produce your custom network, see the Networks Section in the repository.

In [2]:
obj = sc.read("./data/pbmc3k_final.h5ad")
graph_file = 'PPI.csv'

## Model training

Optionally, the parameters of each model can be individually changed. Additionally, if no parameters are given for one of the models, it simply runs with the default values. 

In [3]:
parameters_scnet = dict(
    annotation_file=graph_file,
    pre_processing_flag=False,
    human_flag=False,
    number_of_batches=1,
    split_cells=True,
    max_epoch=300,
    model_name=RUN_NAME,
    clf_loss=False,
)

parameters_scgpt = dict(
    model_name=RUN_NAME,
    seed=0,
    dataset_name=RUN_NAME,
    do_train=True,
    test_size=0.4,
    load_model="./src/scgpt/checkpoints/scGPT_human",
    mask_ratio=0,
    epochs=10,
    n_bins=51,
    MVC=False, # Masked value prediction for cell embedding
    ecs_thres=0.0, # Elastic cell similarity objective, 0.0 to 1.0, 0.0 to disable
    dab_weight=0.0,
    lr=5e-5,
    batch_size=5,
    layer_size=75,
    nlayers=4,  # number of nn.TransformerEncoderLayer in nn.TransformerEncoder
    nhead=4,  # number of heads in nn.MultiheadAttention
    dropout=0.3,  # dropout probability
    schedule_ratio=0.9,  # ratio of epochs for learning rate schedule
    save_eval_interval=5,
    fast_transformer=True,
    pre_norm=False,
    amp=True,  # Automatic Mixed Precision
    include_zero_gene = False,
    freeze = False, #freeze
    DSBN = False,  # Domain-spec batchnorm
)

In this tutorial we only use the Pbmc3k dataset, so the model will train and test on separate partitions of the data according to the *test_size* parameter. There is also the possibility of using a specific training dataset, by using the *training_obj* argument.

Finally, you can train the models by calling the following function, with the option of training both models or just one.

In [4]:
common_scnet_embs, common_scgpt_embs, avg_combined_embs, conq_combined_embs, common_labels = \
    run_scRepresenter(RUN_NAME, obj, graph_file, SAVE_DIR, 300, 10, training_obj=None)

         Falling back to preprocessing with `sc.pp.pca` and default params.


/home/guilherme/scRepresenter/src/scNET/../networks/PPI.csv
N genes: (690, 2638)
Highly variable genes: 690


Training:   0%|          | 1/300 [00:03<19:45,  3.96s/it]

Epoch [1/300]
  Row Loss: 1.7532
  Column Loss: 1.9981
  Classifier Loss: 0.0000
  Final Loss: 5.2922
  AUC: 0.5000


Training:  34%|███▎      | 101/300 [01:20<02:21,  1.41it/s]

Epoch [101/300]
  Row Loss: 0.9725
  Column Loss: 1.7147
  Classifier Loss: 0.0000
  Final Loss: 3.8426
  AUC: 0.7889


Training:  67%|██████▋   | 201/300 [02:29<01:11,  1.39it/s]

Epoch [201/300]
  Row Loss: 0.9276
  Column Loss: 1.6663
  Classifier Loss: 0.0000
  Final Loss: 2.9883
  AUC: 0.7804


Training: 100%|██████████| 300/300 [03:38<00:00,  1.37it/s]


Best Network AUC: 0.795596854444257
save to output/pbmc3k/scGPT
scGPT - INFO - match 1803/2000 genes in vocabulary of size 60697.
scGPT - INFO - Resume model from src/scgpt/checkpoints/scGPT_human/best_model.pt, the model args will override the config src/scgpt/checkpoints/scGPT_human/args.json.
scGPT - INFO - Normalizing total counts ...
scGPT - INFO - Binning data ...
scGPT - INFO - Normalizing total counts ...
scGPT - INFO - Binning data ...
scGPT - INFO - train set number of samples: 1423, 
	 feature length: 392
scGPT - INFO - valid set number of samples: 159, 
	 feature length: 291
scGPT - INFO - Total Pre freeze Params 51335689
scGPT - INFO - Total Post freeze Params 51335689
random masking at epoch   1, ratio of masked values in train:  0.0000
scGPT - INFO - | epoch   1 | 100/285 batches | lr 0.0001 | ms/batch 908.79 | loss  1.68 | cls  1.68 | err  0.56 | 
scGPT - INFO - | epoch   1 | 200/285 batches | lr 0.0001 | ms/batch 897.18 | loss  1.04 | cls  1.04 | err  0.34 | 
scGPT - I